# Truncatiebewuste MX-training: vijf epochs

CNN-8 / SVHN train-only. Hoge modus: MXFP6 E2M3. Lage modus: E2M1 **uit dezelfde E2M3-codes via >> 2**, zonder nieuwe schaalkeuze. A32 en FP32-masters. Dit notebook leest uitsluitend opgeslagen JSON/CSV en voert geen modelcalls, training, downloads of herkalibratie uit.

Een pilot toetst de implementatie en looptijd; verbetering van de nauwkeurigheid staat niet vooraf vast. De oude directe tweemodusrun is de methodologische referentie. Vergelijk op hetzelfde aantal epochs; niet een nieuwe vijf-epochscore met een oude honderd-epochscore.

In [ ]:
from pathlib import Path
RUN_DIR = Path.cwd()
BASELINE_DIR = None  # Optioneel: resultatenmap van de DIRECTE tweemodusrun (vijf of honderd epochs).

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def read_json(name):
    return json.loads((RUN_DIR / name).read_text(encoding="utf-8"))
report = read_json("run_report.json")
assert report["schema"] == "mx-truncation-aware-pilot5-report-v1", "Verkeerde runmap"
assert report["config"]["truncation"] is True
synthetic = report.get("synthetic_fixture_only", False)
if synthetic:
    display(Markdown("**KUNSTMATIGE CPU-TESTDATA: GEEN ONDERZOEKSRESULTATEN.**"))
if report["status"] != "PASS":
    display(Markdown("**Deze run is niet volledig geslaagd: " + report["status"] + ".**"))
metrics = pd.read_csv(RUN_DIR / "epoch_metrics.csv")
selected = pd.read_csv(RUN_DIR / "selected_metrics.csv")
history = read_json("history.json")
summary = read_json("training_summary.json")
assert summary["optimizer_steps"] == summary["completed_epochs"] * ((report['config']['train_examples'] + report['config']['batch_size'] - 1) // report['config']['batch_size'])
assert not metrics.duplicated(["epoch", "phase", "weight_bits"]).any()
assert len(metrics) == summary["completed_epochs"] * 4
assert metrics[["objective_loss","count_top1_percent"]].notna().all().all()
assert ((100*metrics.correct1/metrics.n - metrics.count_top1_percent).abs() < 1e-8).all()
display(pd.DataFrame([{k:summary.get(k) for k in ["completed_epochs","optimizer_steps","training_forward_passes","truncation","measured_completed_epoch_seconds"]}]))
display(selected[["role","epoch","weight_bits","weight_format","bn_bank_key","count_top1_percent","objective_loss"]])

## Leercurves
De testloss is voor beide modi hard-label cross-entropy. De hoge trainingsloss is hard-label CE; de lage trainingsloss is soft-target CE met de losgekoppelde E2M3-teacher. Deze verschillende trainingsdoelen mogen niet als dezelfde grootheid worden behandeld.

In [ ]:
labels = {6:"E2M3/A32",4:"E2M1 uit E2M3-truncatie/A32"}
fig, ax = plt.subplots(figsize=(8,4.2))
for bit in [6,4]:
    d=metrics[(metrics.phase=="test") & (metrics.weight_bits==bit)]
    ax.plot(d.epoch,d.count_top1_percent, marker="o",label=labels[bit])
ax.set(xlabel="Epoch",ylabel="Testaccuracy (%)",title="Kunstmatige testdata" if synthetic else "Testaccuracy")
ax.legend(); ax.grid(alpha=.25); fig.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(8,4.2))
for bit in [6,4]:
    d=metrics[(metrics.phase=="test") & (metrics.weight_bits==bit)]
    ax.plot(d.epoch,d.objective_loss,marker="o",label=labels[bit])
ax.set(xlabel="Epoch",ylabel="Hard-label cross-entropy",title="Testloss")
ax.legend(); ax.grid(alpha=.25); fig.tight_layout(); plt.show()

## Gedeelde bron, gradients en BatchNorm
Elke trainingsminibatch heeft twee forwards en een gezamenlijke Adam-update. De zes hoge laagbronnen worden eenmaal gevormd en voor beide modi gebruikt. Schalen, teken en exponent blijven bij de lage afleiding behouden. De cache wordt voor de optimizerupdate opgeruimd; na een update mag een nieuwe hoge bron ontstaan.

In [ ]:
audit_rows=[]
for e in history["epochs"]:
    g=e["gradient_checks"]; src=g["shared_source_checks"]
    assert src["source_cache_cleared"] and src["no_direct_low_quantizer"]
    assert src["high_encodings"] == src["low_from_same_source"] == 6*src["source_pairs"]
    assert g["detached_teacher_checks"] == g["finite_gradient_checks"] == src["source_pairs"]
    assert e["batchnorm_counters_correct"] and e["test_optimizer_steps"]==0
    test_src=e["test_source_checks"]
    assert test_src["source_cache_cleared"]
    audit_rows.append({"epoch":e["epoch"],"updates":g["optimizer_steps_in_phase"],
        "high_encodings":src["high_encodings"],"low_from_same_high":src["low_from_same_source"],
        "gradient_norm_max":g["gradient_l2_max"],"BN_ok":e["batchnorm_counters_correct"]})
display(pd.DataFrame(audit_rows))
updates=pd.read_csv(RUN_DIR/"weight_updates.csv")
display(updates.groupby(["epoch","weight_bits"])[["changed_element_codes_including_padding","changed_scale_codes"]].sum())
display(Markdown("Master-updatewaarden staan voor beide modi herhaald in het bronbestand: **niet dubbel optellen**."))
display(pd.DataFrame(report.get("warnings",[])))

## Tijdmeting en exports
De ramingen gelden voor deze software-emulator, inclusief controles; niet voor native MX-hardware. De hoge export bevat codes, schalen en de aanvullende laag-/BN-staat, zonder de zes FP32-mastergewichten. Het lage bestand is een controlekopie afgeleid uit dat hoge bestand; het is geen noodzakelijke tweede bron. De FP32-resumecheckpoints worden apart bewaard. Er is geen extra volledige testsetpass vanuit de exports in deze pilot.

In [ ]:
times=pd.read_csv(RUN_DIR/"epoch_times.csv")
estimate=read_json("timing_estimate.json")
display(times[["epoch","train_seconds","test_seconds","total_epoch_seconds"]])
print("Gemeten epochtijd (minuten):", times.total_epoch_seconds.sum()/60)
if estimate.get("available"):
    print("Extrapolatie 100 epochs (minuten):", estimate["estimated_100_epoch_seconds"]/60)
    print("Waargenomen tijdspreiding, GEEN betrouwbaarheidsinterval:", estimate.get("observed_range_projection_seconds"))
display(report.get("mx_exports",{}))

## Optionele vergelijking met directe tweemoduskwantisatie
Dit is alleen zinvol op hetzelfde trainingsbudget. Een verschil kan ook BN-aanpassing aan het truncatiepad omvatten. Een seed levert geen algemene nauwkeurigheidsclaim op. De testgestuurde checkpointselectie is niet onafhankelijk; toon ook de vaste laatste epoch.

In [ ]:
if BASELINE_DIR is not None:
    base=Path(BASELINE_DIR)
    baseline_report=json.loads((base/"run_report.json").read_text(encoding="utf-8"))
    bc=baseline_report["config"]; tc=report["config"]
    assert bc["truncation"] is False
    for key in ["model","expand","train_examples","test_examples","activation_bits","batch_size","optimizer","lr","weight_decay","scheduler_milestones","seed","weight_bits","bn_bank_mapping"]:
        assert tc[key]==bc[key], f"Protocolverschil: {key}"
    if "initial_model_sha256" in baseline_report:
        assert baseline_report["initial_model_sha256"]==report["initial_model_sha256"]
    old=pd.read_csv(base/"epoch_metrics.csv")
    new=metrics[metrics.phase=="test"]
    old=old[old.phase=="test"]
    comparison=new.merge(old,on=["epoch","weight_bits"],suffixes=("_trunc","_direct"),validate="one_to_one")
    comparison["verschil_procentpunt"]=comparison.count_top1_percent_trunc-comparison.count_top1_percent_direct
    display(comparison[["epoch","weight_bits","count_top1_percent_direct","count_top1_percent_trunc","verschil_procentpunt"]])
    display(Markdown("Rapporteer alleen deze gelijk-epochvergelijking; dit is nog geen bewijs van een robuuste verbetering."))
else:
    print("Geen baselinepad ingevuld; alleen deze proef getoond.")